<a href="https://colab.research.google.com/github/Dennischegemwangi/code-base/blob/main/multilingual_named_entity_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#NER
# identifying dataset configuration (domains)
from datasets import get_dataset_config_names
xtreme_subset = get_dataset_config_names("google/xtreme")
print (f"{len(xtreme_subset)} configuration found in xtreme")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

183 configuration found in xtreme


In [ ]:
#identifying PAN-X
panx_sub = [s for s in xtreme_subset if s.startswith ("PAN")]

panx_sub[:3]

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg']

In [ ]:
from collections import defaultdict
from datasets import DatasetDict,load_dataset


langs = ["fr","it","de","en"]
fracs = [0.229,0.084,0.629,0.059]
#return dict if key doesn't mot exit
panx_ch = defaultdict (DatasetDict)

for lang ,frac in zip(langs,fracs):
    ds = load_dataset("google/xtreme" ,name = f"PAN-X.{lang}")


    # shuffle and down sample each split according to spoken proportion

    for split in ds:
        panx_ch[lang][split] = (ds[split]. shuffle (seed = 0).select(range(int(frac * ds[split].num_rows))))

PAN-X.fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  837kB            

PAN-X.fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PAN-X.fr/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  419kB            

PAN-X.fr/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

PAN-X.fr/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  423kB            

PAN-X.fr/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (26 outputs left out)

PAN-X.en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd
pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs },index =["num_training_examples"])

,fr,it,de,en
num_training_examples,4580,1680,12580,1180


In [ ]:
#performing zero-shot cross lingual transfer

element = panx_ch["de"]["train"][0]
#for key,value in element.items():
    #print(f"{key}:{value}")

#features of the dataset
for key, value in panx_ch["de"]["train"].features.items():
    print(f"{key}:{value}")

tokens:List(Value('string'))
ner_tags:List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs:List(Value('string'))


In [ ]:
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print (tags)

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [ ]:
#creating new column within the dataset for NER_TAGS that are easy to familiarize

def create_tag_names (batch):
    return {"ner_tags_str" : [tags.int2str(idx) for idx in batch["ner_tags"]]}

panx_de = panx_ch["de"].map(create_tag_names)



de_example = panx_de["train"][0]

pd.DataFrame([de_example["tokens"],de_example["ner_tags_str"]],['Tokens','Tags'])

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [ ]:
#checking for imbalance within the tags

from collections import Counter


split2freq = defaultdict (Counter)

for split, dataset in panx_de.items():
    for row in dataset["ner_tags_str"]:
        for tag in row:
            if tag.startswith("B"):

               tag_type = tag.split("-")[1]
               split2freq[split][tag_type] +=1

pd.DataFrame.from_dict(split2freq,orient = "index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


In [ ]:
#wordpiece vs sentencepiece Tokenization


from transformers import AutoTokenizer


bert = "bert-base-cased"
xlm_r = "xlm-roberta-base"

bert_tokenizer = AutoTokenizer.from_pretrained (bert)
xlm_r_tokenizer = AutoTokenizer.from_pretrained(xlm_r)


text = "Dennis visited Nairobi National park!"

bert_token = bert_tokenizer(text).tokens()
xlm_r_token = xlm_r_tokenizer (text).tokens()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
print (bert_token)
print (xlm_r_token)

['[CLS]', 'Dennis', 'visited', 'Nairobi', 'National', 'park', '!', '[SEP]']
['<s>', '▁Dennis', '▁visite', 'd', '▁Nairobi', '▁National', '▁park', '!', '</s>']


In [ ]:
"".join(xlm_r_token).replace (u"\u2581", " ")

'<s> Dennis visited Nairobi National park!</s>'

In [ ]:
# Transformer for named entity recognition
# creating a custom head for token classification

import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers import XLMRobertaPreTrainedModel


class XlmRobertaForTokenClassification(XLMRobertaPreTrainedModel):
    config_class = XLMRobertaConfig
    # base_model_prefix tells XLMRobertaPreTrainedModel that self.roberta holds the base weights
    base_model_prefix = "roberta"

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # load the model's body
        self.roberta = RobertaModel(config, add_pooling_layer=False)

        # set up the Token Classification head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # weights initialization
        self.post_init()

    def forward(
        self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs
    ):
        # use model's body to acquire hidden states
        outputs = self.roberta(
            input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, **kwargs
        )
        # apply hidden states to the classifier
        seq_output = self.dropout(outputs[0])
        logits = self.classifier(seq_output)
        # calculate loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
#loading the model
indx2tag = {idx: tag for idx,tag in enumerate (tags.names)}
tag2indx = {tag: idx for tag , idx in enumerate (tags.names)}

In [ ]:
from transformers import AutoConfig

xlmr_config = AutoConfig.from_pretrained(xlm_r,num_labels = tags.num_classes,id2label =indx2tag,label2id = tag2indx)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate your custom model
my_xlmr_model = XlmRobertaForTokenClassification.from_pretrained(xlm_r, config=xlmr_config).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XlmRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
input_ids =  xlm_r_tokenizer.encode(text, return_tensors = "pt")

pd.DataFrame([xlm_r_token, input_ids[0].numpy()],index = ["Tokens","ids"])

,0,1,2,3,4,5,6,7,8
Tokens,<s>,▁Dennis,▁visite,d,▁Nairobi,▁National,▁park,!,</s>
ids,0,124748,36997,71,116887,9907,9201,38,2


In [ ]:
outputs = my_xlmr_model(input_ids.to(device)).logits
predictions = torch.argmax(outputs, dim=-1)

print(f"number of tokens in the sequence: {len(xlm_r_token)}")
print(f"output shape : {outputs.shape}")

number of tokens in the sequence: 9
output shape : torch.Size([1, 9, 7])


In [ ]:
preds =[tags.names[p] for p in predictions[0].cpu().numpy()]

pd.DataFrame([xlm_r_token,preds],index = ["Tokens","tags"])

,0,1,2,3,4,5,6,7,8
Tokens,<s>,▁Dennis,▁visite,d,▁Nairobi,▁National,▁park,!,</s>
tags,B-PER,B-PER,B-PER,B-PER,B-PER,B-PER,B-PER,B-PER,B-PER


In [ ]:
#creating a helper function that wraps everything together

def tag_text(text,tags, model, tokenizer):
    tokens = tokenizer(text).tokens()
    input_ids =  xlm_r_tokenizer.encode(text, return_tensors = "pt")
    outputs= model(inputs)[0]
    predictions = torch.argmax(outputs,dim = -2)
    preds =[tags.names[p] for p in predictions[0].cpu().numpy()]
    return pd.DataFrame([xlm_r_token,preds],index = ["Tokens","tags"])

In [ ]:
#Tokenizing the dataset for NER

word,label = de_example["tokens"],de_example["ner_tags"]


tokenized_input = xlm_r_tokenizer(de_example["tokens"],is_split_into_words = True)

tokens = xlm_r_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
pd.DataFrame([tokens], index = ["Tokens"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>


In [ ]:
#masking subwords
word_ids = tokenized_input.word_ids()
pd.DataFrame([tokens,word_ids],index = ["Tokens","word ids"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
word ids,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None


In [30]:
previous_word_idx = None
label_ids =[]

for word_idx in word_ids:
    if word_idx is  None  or word_idx == previous_word_idx:
       label_ids.append(-100)
    elif word_idx != previous_word_idx:
         label_ids.append(label[word_idx])
    previous_word_idx = word_idx
labels = [indx2tag[l] if l!=-100 else "IGN" for l in label_ids]

index = ["Tokens","word_ids","label_ids","labels"]


pd.DataFrame([tokens,word_ids,label_ids,labels],index = index)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
word_ids,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None
label_ids,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
labels,IGN,O,O,IGN,O,O,B-LOC,IGN,IGN,I-LOC,...,B-LOC,IGN,IGN,IGN,I-LOC,IGN,IGN,O,IGN,IGN


In [33]:
def tokenize_and_align_labels(examples):
    tokenized_input = xlm_r_tokenizer(examples["tokens"],trancation = True,is_split_into_words = True)
    labels = []

    for idx,label in enumerate (examples["ner_tags"]):
        word_ids = tokenized_input.word_ids(batch_index = idx)
        previous_word_idx = None

        label_ids =[]

        for word_idx in word_ids:
            if word_idx is  None  or word_idx == previous_word_idx:
               label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_input["labels"] = labels

    return tokenized_input

In [34]:
def encode_panx_dataset(corpus):
    return corpus.map(tokenize_and_align_labels,batched = True,remove_columns = ['langs','ner_tags', 'tokens'])

In [35]:
panx_de_encoded = encode_panx_dataset(panx_ch["de"])

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]